# CFPB Complaints: Preprocessing & Feature Engineering

This notebook is the single source of truth for modeling-ready data preparation.

**Main purpose:** define the final target label and build leakage-safe features used by all downstream models.

**Target:** Predict whether a complaint receives monetary relief (binary classification).
- 1: Monetary relief given (e.g., `Closed with monetary relief`)
- 0: No monetary relief (e.g., `Closed with explanation`, `Closed`)

**Output artifacts:**
- `data/processed/train_features.csv`
- `data/processed/test_features.csv`

**Boundary:** EDA visuals stay in notebook 01; model training/evaluation stay in notebook 03.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from datetime import datetime

# Load the sampled data
sample_file = Path('../data/processed/complaints_sample.csv')
df = pd.read_csv(sample_file)

print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Define the binary target: monetary relief vs. no monetary relief
print('Unique company responses:')
print(df['Company response to consumer'].value_counts(dropna=False))

# Create target: 1 if 'Monetary relief' is mentioned, 0 otherwise
df['target_monetary_relief'] = (
    df['Company response to consumer'].str.contains('Monetary relief', case=False, na=False).astype(int)
)

print(f'\nTarget distribution:')
print(df['target_monetary_relief'].value_counts())
print(f'Class balance: {df["target_monetary_relief"].mean():.2%} positive')

In [ ]:
# Helper function to clean narrative text
def clean_narrative(text):
    if pd.isna(text):
        return ''
    # Lowercase
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove phone-like patterns (XXXX patterns from narrative scrubbing)
    text = re.sub(r'\bx{2,}\b', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Clean narratives
df['narrative_clean'] = df['Consumer complaint narrative'].apply(clean_narrative)
df['narrative_length'] = df['narrative_clean'].str.len()
df['has_narrative'] = (df['narrative_length'] > 0).astype(int)

print(f'Narratives present: {df["has_narrative"].sum()} / {len(df)}')
print(f'Narrative length (mean): {df[df["has_narrative"] == 1]["narrative_length"].mean():.0f} chars')

In [ ]:
# Extract temporal features
df['Date received'] = pd.to_datetime(df['Date received'], errors='coerce')
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'], errors='coerce')

df['year_received'] = df['Date received'].dt.year
df['month_received'] = df['Date received'].dt.month
df['quarter_received'] = df['Date received'].dt.quarter
df['days_to_send'] = (df['Date sent to company'] - df['Date received']).dt.days

print('Temporal features created:')
print(f'  year_received: {df["year_received"].unique()}')
print(f'  days_to_send (mean): {df["days_to_send"].mean():.1f}')

In [ ]:
# Encode categorical features
categorical_features = ['Product', 'Issue', 'State', 'Submitted via']

# One-hot encode (we'll use label encoding later in the pipeline for tree models)
df_cat_encoded = pd.get_dummies(df[categorical_features], prefix=categorical_features, drop_first=False)

print(f'Categorical features encoded:')
print(f'  Total new columns: {len(df_cat_encoded.columns)}')
print(f'  Sample columns: {list(df_cat_encoded.columns)[:5]}')

In [ ]:
# Build the final feature set
# Numeric features
numeric_features = ['narrative_length', 'has_narrative', 'year_received', 'month_received', 'quarter_received']
X_numeric = df[numeric_features].fillna(0)

# Combine
X = pd.concat([X_numeric, df_cat_encoded, df[['narrative_clean']]], axis=1)
y = df['target_monetary_relief']

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Target distribution:\n{y.value_counts()}')

In [ ]:
# Train-test split (80-20)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape[0]:,} rows ({y_train.mean():.2%} positive)')
print(f'Test set: {X_test.shape[0]:,} rows ({y_test.mean():.2%} positive)')

# Save train/test split for modeling notebooks
train_data = X_train.copy()
train_data['target'] = y_train.values
test_data = X_test.copy()
test_data['target'] = y_test.values

train_data.to_csv('../data/processed/train_features.csv', index=False)
test_data.to_csv('../data/processed/test_features.csv', index=False)

print(f'\nSaved train/test splits to data/processed/')

In [ ]:
# Summary
print("\n" + "="*60)
print("PREPROCESSING COMPLETE")
print("="*60)
print(f"Target: Monetary relief (binary classification)")
print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print(f"Features: {X.shape[1]}")
print(f"  - Narratives with text: {X['narrative_clean'].str.len().gt(0).sum()}")
print(f"  - Categorical features encoded: {len(df_cat_encoded.columns)}")
print(f"  - Numeric/temporal features: {len(numeric_features)}")
print(f"\nNext: Run 03_modeling.ipynb for model comparison")